# **Baseline Notebook**



---
## Setup Environment

In [40]:
# DO NOT MODIFY THE CODE IN THIS CELL
!pip install -q utstd

from utstd.folders import *
from utstd.ipyrenders import *

at = AtFolder(
    course_code=36106,
    assignment="AT3",
)
at.run()

import warnings
warnings.simplefilter(action='ignore')


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip



You can now save your data files in: /Users/aryan/Machine Learning Assignment 3/36106/assignment/AT3/data


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
utstd 0.1.8 requires scikit-learn~=1.5.1, but you have scikit-learn 1.6.1 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
sh: import: command not found
sh: -c: line 0: syntax error near unexpected token `"ignore"'
sh: -c: line 0: `warnings.filterwarnings("ignore")'


---
## Student Information

In [41]:
group_name = "36106-26AU-AT3-Group01"
student_name = "Aryan Goel"
student_id = "26040826"

In [42]:
# Do not modify this code
print_tile(size="h1", key='group_name', value=group_name)

In [43]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h1", key='student_name', value=student_name)

In [44]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h1", key='student_id', value=student_id)

---
## 0. Python Packages

### 0.a Install Additional Packages

> If you are using additional packages, you need to install them here using the command: `! pip install <package_name>`

In [45]:
# <Student to fill this section and then remove this comment>

### 0.b Import Packages

In [46]:
# <Student to fill this section and then remove this comment>
import pandas as pd
import altair as alt

---
## A. Assess Baseline Model

In [47]:
# DO NOT MODIFY THE CODE IN THIS CELL
# Load data
try:
  X_train = pd.read_csv(at.folder_path / 'X_train.csv')
  y_train = pd.read_csv(at.folder_path / 'y_train.csv')

  X_val = pd.read_csv(at.folder_path / 'X_val.csv')
  y_val = pd.read_csv(at.folder_path / 'y_val.csv')

  X_test = pd.read_csv(at.folder_path / 'X_test.csv')
  y_test = pd.read_csv(at.folder_path / 'y_test.csv')
except Exception as e:
  print(e)

### A.1 Generate Predictions with Baseline Model

In [48]:
# A.1 Generate Predictions with Baseline Model
# Baseline choice: "Most Frequent Class" (DummyClassifier strategy='most_frequent')
# Why: provides a minimal benchmark for classification (beats random guessing in imbalanced data).

from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd
import numpy as np

# --- Ensure y is 1D (Series) ---
def to_1d(y):
    if isinstance(y, pd.DataFrame):
        if y.shape[1] != 1:
            raise ValueError(f"Expected y to have 1 column, got {y.shape[1]}")
        return y.iloc[:, 0]
    return pd.Series(y)

y_train_s = to_1d(y_train)
y_val_s   = to_1d(y_val)
y_test_s  = to_1d(y_test)

# --- Fit baseline model on TRAIN only ---
baseline_clf = DummyClassifier(strategy="most_frequent", random_state=42)
baseline_clf.fit(X_train, y_train_s)

# --- Predictions ---
y_train_pred = baseline_clf.predict(X_train)
y_val_pred   = baseline_clf.predict(X_val)
y_test_pred  = baseline_clf.predict(X_test)

# --- Predicted probabilities (if needed later) ---
# DummyClassifier supports predict_proba for most strategies.
y_val_proba = baseline_clf.predict_proba(X_val)[:, 1] if hasattr(baseline_clf, "predict_proba") else None
y_test_proba = baseline_clf.predict_proba(X_test)[:, 1] if hasattr(baseline_clf, "predict_proba") else None

print("Baseline model fitted (most_frequent).")
print("Most frequent class in TRAIN:", int(y_train_s.mode().iloc[0]))
print("Train prediction unique values:", np.unique(y_train_pred))
print("Val prediction unique values:  ", np.unique(y_val_pred))
print("Test prediction unique values: ", np.unique(y_test_pred))

Baseline model fitted (most_frequent).
Most frequent class in TRAIN: 0
Train prediction unique values: [0]
Val prediction unique values:   [0]
Test prediction unique values:  [0]


In [49]:
import pandas as pd
import numpy as np

def to_1d(y):
    if isinstance(y, pd.DataFrame):
        if y.shape[1] != 1:
            raise ValueError(f"Expected y to have 1 column, got {y.shape[1]}")
        return y.iloc[:, 0]
    return pd.Series(y)

y_train_s = to_1d(y_train)
y_val_s   = to_1d(y_val)
y_test_s  = to_1d(y_test)

def class_balance(y, name):
    vc = pd.Series(y).value_counts(dropna=False).sort_index()
    pct = (vc / vc.sum()).round(4)
    out = pd.DataFrame({"count": vc, "percent": pct})
    out.index.name = f"{name}_class"
    return out

print("Class balance (TRAIN):")
display(class_balance(y_train_s, "train"))

print("Class balance (VAL):")
display(class_balance(y_val_s, "val"))

print("Class balance (TEST):")
display(class_balance(y_test_s, "test"))

# Expected accuracy if always predicting the most frequent class in TRAIN
majority_class = int(pd.Series(y_train_s).mode().iloc[0])
expected_acc_train = float((y_train_s == majority_class).mean())
expected_acc_val   = float((y_val_s == majority_class).mean())
expected_acc_test  = float((y_test_s == majority_class).mean())

print("Majority class (TRAIN):", majority_class)
print("Expected accuracy if always predict majority class:")
print("  train:", round(expected_acc_train, 4))
print("  val:  ", round(expected_acc_val, 4))
print("  test: ", round(expected_acc_test, 4))

Class balance (TRAIN):


,count,percent
train_class,,
0,12080,0.9026
1,1303,0.0974


Class balance (VAL):


,count,percent
val_class,,
0,2598,0.9059
1,270,0.0941


Class balance (TEST):


,count,percent
test_class,,
0,2562,0.8933
1,306,0.1067


Majority class (TRAIN): 0
Expected accuracy if always predict majority class:
  train: 0.9026
  val:   0.9059
  test:  0.8933


In [50]:
# A.1 EXTRA 2: Second baseline model (stratified random guessing)
# This baseline predicts classes according to their frequency in TRAIN.

from sklearn.dummy import DummyClassifier
import numpy as np

random_baseline = DummyClassifier(strategy="stratified", random_state=42)
random_baseline.fit(X_train, y_train_s)

y_val_pred_strat = random_baseline.predict(X_val)
y_test_pred_strat = random_baseline.predict(X_test)

y_val_proba_strat = random_baseline.predict_proba(X_val)[:, 1]
y_test_proba_strat = random_baseline.predict_proba(X_test)[:, 1]

print("Stratified baseline created.")
print("Unique predictions (VAL):", np.unique(y_val_pred_strat))

Stratified baseline created.
Unique predictions (VAL): [0 1]


### A.2 Selection of Performance Metrics

> Provide some explanations on why you believe the performance metrics you chose is appropriate


In [51]:
# A.2 Performance Metrics (compute and display)
# We report: Accuracy, Precision, Recall, F1, ROC-AUC (when proba available)
# Motivation: classification problems can be imbalanced; F1/Recall/Precision are more informative than accuracy alone.

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)
import pandas as pd
import numpy as np

def safe_roc_auc(y_true, y_proba):
    if y_proba is None:
        return np.nan
    # roc_auc requires both classes present
    if len(pd.Series(y_true).unique()) < 2:
        return np.nan
    return roc_auc_score(y_true, y_proba)

def metric_row(split_name, y_true, y_pred, y_proba=None):
    return {
        "split": split_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": safe_roc_auc(y_true, y_proba),
    }

metrics_df = pd.DataFrame([
    metric_row("train", y_train_s, y_train_pred, baseline_clf.predict_proba(X_train)[:,1]),
    metric_row("val",   y_val_s,   y_val_pred,   y_val_proba),
    metric_row("test",  y_test_s,  y_test_pred,  y_test_proba),
])

display(metrics_df)

print("\nConfusion matrix (VAL):")
display(pd.DataFrame(confusion_matrix(y_val_s, y_val_pred),
                     index=["Actual 0", "Actual 1"],
                     columns=["Pred 0", "Pred 1"]))

,split,accuracy,precision,recall,f1,roc_auc
0,train,0.902638,0.0,0.0,0.0,0.5
1,val,0.905858,0.0,0.0,0.0,0.5
2,test,0.893305,0.0,0.0,0.0,0.5



Confusion matrix (VAL):


,Pred 0,Pred 1
Actual 0,2598,0
Actual 1,270,0


In [52]:
# A.2 EXTRA 1: Reusable evaluation function + compare baselines side-by-side

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score
)
import pandas as pd
import numpy as np

def eval_binary(y_true, y_pred, y_proba=None, label="model", split="val"):
    y_true = pd.Series(y_true)

    out = {
        "model": label,
        "split": split,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": np.nan,
        "avg_precision(PR-AUC)": np.nan,
    }

    # Only compute AUC metrics if we have probabilities and both classes exist
    if y_proba is not None and y_true.nunique() == 2:
        out["roc_auc"] = roc_auc_score(y_true, y_proba)
        out["avg_precision(PR-AUC)"] = average_precision_score(y_true, y_proba)

    return out

rows = []

# Baseline 1: most_frequent (from A.1)
rows.append(eval_binary(y_val_s, y_val_pred, y_val_proba, label="most_frequent", split="val"))
rows.append(eval_binary(y_test_s, y_test_pred, y_test_proba, label="most_frequent", split="test"))

# Baseline 2: stratified (if you ran A.1 Extra Cell 2)
if "y_val_pred_strat" in globals():
    rows.append(eval_binary(y_val_s, y_val_pred_strat, y_val_proba_strat, label="stratified", split="val"))
    rows.append(eval_binary(y_test_s, y_test_pred_strat, y_test_proba_strat, label="stratified", split="test"))

metrics_compare = pd.DataFrame(rows).sort_values(["split", "model"])
display(metrics_compare)

,model,split,accuracy,precision,recall,f1,roc_auc,avg_precision(PR-AUC)
1,most_frequent,test,0.893305,0.000000,0.000000,0.000000,0.500000,0.106695
3,stratified,test,0.816597,0.101449,0.091503,0.096220,0.497352,0.106215
0,most_frequent,val,0.905858,0.000000,0.000000,0.000000,0.500000,0.094142
2,stratified,val,0.828452,0.097826,0.100000,0.098901,0.502079,0.094511


In [53]:
# A.2 EXTRA 2: Confusion matrix with derived rates (VAL)

from sklearn.metrics import confusion_matrix
import pandas as pd
import numpy as np

def confusion_details(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    # Rates
    tpr = tp / (tp + fn) if (tp + fn) else np.nan  # recall / sensitivity
    fpr = fp / (fp + tn) if (fp + tn) else np.nan
    ppv = tp / (tp + fp) if (tp + fp) else np.nan  # precision
    npv = tn / (tn + fn) if (tn + fn) else np.nan
    return {"tn": tn, "fp": fp, "fn": fn, "tp": tp, "TPR(recall)": tpr, "FPR": fpr, "PPV(precision)": ppv, "NPV": npv}

val_details = confusion_details(y_val_s, y_val_pred)
display(pd.DataFrame([val_details]))

print("Confusion matrix (VAL) as table:")
display(pd.DataFrame(confusion_matrix(y_val_s, y_val_pred),
                     index=["Actual 0", "Actual 1"],
                     columns=["Pred 0", "Pred 1"]))

,tn,fp,fn,tp,TPR(recall),FPR,PPV(precision),NPV
0,2598,0,270,0,0.0,0.0,NaN,0.905858


Confusion matrix (VAL) as table:


,Pred 0,Pred 1
Actual 0,2598,0
Actual 1,270,0


In [54]:
# A.2 EXTRA 3: Threshold sweep (VAL) to see trade-off between precision and recall

import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

if y_val_proba is None:
    print("No probabilities available for threshold sweep.")
else:
    thresholds = np.linspace(0.05, 0.95, 19)
    rows = []
    for t in thresholds:
        pred_t = (y_val_proba >= t).astype(int)
        rows.append({
            "threshold": t,
            "precision": precision_score(y_val_s, pred_t, zero_division=0),
            "recall": recall_score(y_val_s, pred_t, zero_division=0),
            "f1": f1_score(y_val_s, pred_t, zero_division=0),
        })
    thr_df = pd.DataFrame(rows)
    display(thr_df)

    # Best threshold by F1
    best = thr_df.sort_values("f1", ascending=False).head(1)
    print("Best threshold on VAL by F1:")
    display(best)

,threshold,precision,recall,f1
0,0.05,0.0,0.0,0.0
1,0.10,0.0,0.0,0.0
2,0.15,0.0,0.0,0.0
3,0.20,0.0,0.0,0.0
4,0.25,0.0,0.0,0.0
5,0.30,0.0,0.0,0.0
6,0.35,0.0,0.0,0.0
7,0.40,0.0,0.0,0.0
8,0.45,0.0,0.0,0.0
9,0.50,0.0,0.0,0.0


Best threshold on VAL by F1:


,threshold,precision,recall,f1
0,0.05,0.0,0.0,0.0


In [55]:
# <Student to fill this section and then remove this comment>
performance_metrics_explanations = """
I selected multiple classification performance metrics (Accuracy, Precision, Recall, F1-score, and PR-AUC / ROC-AUC)
because relying on a single metric can be misleading, especially when the target classes are imbalanced.

Why accuracy alone is not sufficient
- In our dataset the positive class rate is low (around ~10%). This means a naive model can predict the majority class
  (e.g., always predict 0) and still achieve high accuracy (~0.90), while completely failing to identify any positives.
- This is exactly what we observed with the 'most_frequent' baseline: high accuracy but precision/recall/F1 of 0 for the
  positive class, so the model is not actionable for the business.

Why precision is important
- Precision answers: “When the model predicts a positive outcome, how often is it correct?”
- This matters when acting on predicted positives has a cost (e.g., sending discounts/marketing offers, contacting customers,
  allocating limited resources). Low precision would waste budget and effort on false positives.

Why recall is important
- Recall answers: “Out of all true positive cases, how many did we successfully detect?”
- This matters when missing a positive is costly (e.g., missed opportunity to retain a customer who would otherwise churn,
  failing to identify customers likely to reorder, etc.). For many retail use cases, improving recall for the positive class
  creates direct business value.

Why F1-score is important
- F1-score balances precision and recall in a single number.
- It is useful when classes are imbalanced and we want a metric that penalises models that do well on only one side
  (e.g., high precision but very low recall, or the opposite).

Why PR-AUC (Average Precision) and ROC-AUC are useful
- ROC-AUC measures the model’s ability to rank positive examples above negative examples across all thresholds. However,
  ROC-AUC can look acceptable even when the positive class is rare.
- PR-AUC (Average Precision) is often more informative for imbalanced classification because it focuses on performance on the
  positive class (precision-recall trade-off). In business terms, it better reflects how well the model can identify a small,
  valuable positive group.

Overall justification
- Using this set of metrics gives a fair and business-relevant evaluation:
  - Accuracy confirms general correctness,
  - Precision controls false-positive cost,
  - Recall controls false-negative risk,
  - F1 balances the trade-off,
  - PR-AUC/ROC-AUC evaluate ranking quality and support future threshold selection based on business constraints.
"""

In [56]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='performance_metrics_explanations', value=performance_metrics_explanations)

### A.3 Baseline Model Performance

> Provide some explanations on model performance


In [57]:
# A.3 Baseline Model Performance (detailed)
# Show classification report and highlight why this baseline is weak/strong depending on imbalance.

from sklearn.metrics import classification_report
import pandas as pd

print("Classification report (VAL):")
print(classification_report(y_val_s, y_val_pred, digits=4, zero_division=0))

print("\nClassification report (TEST):")
print(classification_report(y_test_s, y_test_pred, digits=4, zero_division=0))

# Show the positive class rate in each split (helps interpret baseline)
def positive_rate(y):
    y = pd.Series(y)
    return float((y == 1).mean())

rates = pd.DataFrame({
    "split": ["train", "val", "test"],
    "positive_rate": [positive_rate(y_train_s), positive_rate(y_val_s), positive_rate(y_test_s)]
})

display(rates)

print("\nInterpretation helper:")
print("- If positive_rate is low, predicting all zeros can look good on accuracy but will have recall=0 for class 1.")

Classification report (VAL):
              precision    recall  f1-score   support

           0     0.9059    1.0000    0.9506      2598
           1     0.0000    0.0000    0.0000       270

    accuracy                         0.9059      2868
   macro avg     0.4529    0.5000    0.4753      2868
weighted avg     0.8206    0.9059    0.8611      2868


Classification report (TEST):
              precision    recall  f1-score   support

           0     0.8933    1.0000    0.9436      2562
           1     0.0000    0.0000    0.0000       306

    accuracy                         0.8933      2868
   macro avg     0.4467    0.5000    0.4718      2868
weighted avg     0.7980    0.8933    0.8430      2868



,split,positive_rate
0,train,0.097362
1,val,0.094142
2,test,0.106695



Interpretation helper:
- If positive_rate is low, predicting all zeros can look good on accuracy but will have recall=0 for class 1.


In [58]:
# A.3 Baseline Model Performance (enhanced + more evidence)
# - Shows classification report + confusion matrix
# - Adds balanced accuracy and a "majority-class accuracy" benchmark
# - Explains what the baseline is doing under class imbalance

import pandas as pd
import numpy as np
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# --- Helper: ensure y is 1D ---
def to_1d(y):
    if isinstance(y, pd.DataFrame):
        if y.shape[1] != 1:
            raise ValueError(f"Expected y to have 1 column, got {y.shape[1]}")
        return y.iloc[:, 0]
    return pd.Series(y)

y_train_s = to_1d(y_train)
y_val_s   = to_1d(y_val)
y_test_s  = to_1d(y_test)

# (Assumes you already ran A.1 and have y_train_pred/y_val_pred/y_test_pred)
if "y_val_pred" not in globals():
    raise ValueError("Run A.1 first to create baseline predictions (y_val_pred/y_test_pred).")

# --- Positive class rates (imbalance evidence) ---
def positive_rate(y):
    y = pd.Series(y)
    return float((y == 1).mean())

rates = pd.DataFrame({
    "split": ["train", "val", "test"],
    "n_rows": [len(y_train_s), len(y_val_s), len(y_test_s)],
    "positive_rate": [positive_rate(y_train_s), positive_rate(y_val_s), positive_rate(y_test_s)]
})
print("Class imbalance summary:")
display(rates)

# --- Majority class benchmark ---
majority_class = int(pd.Series(y_train_s).mode().iloc[0])
maj_acc_val = float((y_val_s == majority_class).mean())
maj_acc_test = float((y_test_s == majority_class).mean())

print("Majority class in TRAIN:", majority_class)
print("Accuracy if always predicting TRAIN majority class:")
print("  val: ", round(maj_acc_val, 6))
print("  test:", round(maj_acc_test, 6))

# --- Metrics summary function ---
def summary_metrics(y_true, y_pred, split):
    return pd.Series({
        "split": split,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision(pos=1)": precision_score(y_true, y_pred, zero_division=0),
        "recall(pos=1)": recall_score(y_true, y_pred, zero_division=0),
        "f1(pos=1)": f1_score(y_true, y_pred, zero_division=0),
    })

metrics_val = summary_metrics(y_val_s, y_val_pred, "val")
metrics_test = summary_metrics(y_test_s, y_test_pred, "test")

print("\nBaseline performance summary:")
display(pd.DataFrame([metrics_val, metrics_test]))

# --- Confusion matrices ---
def cm_df(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    return pd.DataFrame(cm, index=["Actual 0", "Actual 1"], columns=["Pred 0", "Pred 1"])

print("\nConfusion matrix (VAL):")
display(cm_df(y_val_s, y_val_pred, "VAL"))

print("Confusion matrix (TEST):")
display(cm_df(y_test_s, y_test_pred, "TEST"))

print("\nInterpretation:")
print("- This baseline predicts only the majority class, so it can achieve high accuracy under imbalance.")
print("- However, it typically gives recall=0 for the positive class (no positives detected), meaning it is not actionable.")
print("- Balanced accuracy helps reveal this issue because it averages recall across both classes.")

Class imbalance summary:


,split,n_rows,positive_rate
0,train,13383,0.097362
1,val,2868,0.094142
2,test,2868,0.106695


Majority class in TRAIN: 0
Accuracy if always predicting TRAIN majority class:
  val:  0.905858
  test: 0.893305

Baseline performance summary:


,split,accuracy,balanced_accuracy,precision(pos=1),recall(pos=1),f1(pos=1)
0,val,0.905858,0.5,0.0,0.0,0.0
1,test,0.893305,0.5,0.0,0.0,0.0



Confusion matrix (VAL):


,Pred 0,Pred 1
Actual 0,2598,0
Actual 1,270,0


Confusion matrix (TEST):


,Pred 0,Pred 1
Actual 0,2562,0
Actual 1,306,0



Interpretation:
- This baseline predicts only the majority class, so it can achieve high accuracy under imbalance.
- However, it typically gives recall=0 for the positive class (no positives detected), meaning it is not actionable.
- Balanced accuracy helps reveal this issue because it averages recall across both classes.


In [59]:
# <Student to fill this section and then remove this comment>
baseline_performance_explanations = """
Provide some explanations on model performance
"""

In [60]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='baseline_performance_explanations', value=baseline_performance_explanations)